# Exploración de Incidentes — PISST
**Objetivo:** Explorar los datos de incidentes para validar la lógica que irá a `analytics_service.py`

Este notebook es de solo lectura — nunca escribe en la BD.

## 1. Configuración e imports

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Asegurar que el path apunte a la raíz del proyecto
sys.path.insert(0, os.path.abspath('..'))

load_dotenv('../.env')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Pandas:', pd.__version__)
print('NumPy:', np.__version__)

## 2. Conexión a la BD

In [ ]:
DATABASE_URL = os.getenv('DATABASE_URL')
print('Conectando a:', DATABASE_URL[:40], '...')

engine = create_engine(DATABASE_URL)

# Verificar conexión
with engine.connect() as conn:
    result = conn.execute(text('SELECT COUNT(*) FROM incidentes'))
    total = result.scalar()
    print(f'Total incidentes en BD: {total}')

## 3. Carga de datos (Ingesta)

In [ ]:
query = """
SELECT
    i.id,
    i.tipo,
    i.severidad,
    i.estado,
    i.fecha,
    i.empresa_id,
    a.nombre AS area_nombre
FROM incidentes i
LEFT JOIN lesiones l ON l.incidente_id = i.id
LEFT JOIN areas a    ON a.id = l.area_id
"""

df = pd.read_sql(query, engine)
print(f'Registros cargados: {len(df)}')
df.head()

## 4. Inspección estructural

In [ ]:
print('Shape:', df.shape)
print()
df.info()

In [ ]:
print('Valores nulos por columna:')
print(df.isnull().sum())

## 5. Análisis descriptivo

In [ ]:
print('Distribución por tipo:')
print(df['tipo'].value_counts())
print()
print('Distribución por severidad:')
print(df['severidad'].value_counts())
print()
print('Distribución por estado:')
print(df['estado'].value_counts())

## 6. Transformaciones y cálculos (lógica del servicio)

In [ ]:
if len(df) == 0:
    print('Sin datos — el servicio retornaría ceros.')
else:
    df['fecha'] = pd.to_datetime(df['fecha'])

    # Distribuciones
    por_tipo = df['tipo'].value_counts().to_dict()
    por_severidad = df['severidad'].value_counts().to_dict()

    # Tasa mensual promedio
    meses_activos = df['fecha'].dt.to_period('M').nunique()
    tasa_mensual = round(len(df) / max(meses_activos, 1), 1)

    # Top 3 áreas
    top_areas = df['area_nombre'].dropna().value_counts().head(3).index.tolist()

    # Tendencia — comparar último mes vs mes anterior
    df_sorted = df.sort_values('fecha')
    ultimo_mes = df_sorted['fecha'].max().to_period('M')
    mes_anterior = (ultimo_mes - 1)
    ultimo = len(df[df['fecha'].dt.to_period('M') == ultimo_mes])
    anterior = len(df[df['fecha'].dt.to_period('M') == mes_anterior])
    tendencia = 'estable' if anterior == 0 else ('aumento' if ultimo > anterior * 1.2 else ('baja' if ultimo < anterior * 0.8 else 'estable'))

    resultado = {
        'total_incidentes': len(df),
        'por_tipo': por_tipo,
        'por_severidad': por_severidad,
        'tasa_mensual_promedio': tasa_mensual,
        'top_areas': top_areas,
        'tendencia': tendencia,
    }

    import json
    print(json.dumps(resultado, indent=2, ensure_ascii=False))

## 7. Exportación de hallazgos

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

if len(df) > 0:
    df.to_csv('../data/processed/incidentes_explorados.csv', index=False)
    print('Exportado a data/processed/incidentes_explorados.csv')

## 8. Conclusiones para el servicio

- La función `analizar_incidentes()` puede usar `value_counts()` directamente sobre el DataFrame
- La tasa mensual se calcula dividiendo `len(df)` entre el número de meses distintos con al menos un incidente
- La tendencia compara el último mes calendar vs el anterior (umbral ±20%)
- Si no hay datos, el servicio retorna ceros sin levantar excepción